## Day 5: Robustness, Final Tables, and Polished Figures

This notebook consolidates publishable outputs:
1) quarterly robustness metrics,
2) final model performance table,
3) curated figure set,
4) feature-importance interpretation notes,
5) key findings bullets for the paper draft.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error


PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name.startswith("notebooks") else Path.cwd().resolve()
DATA_DIR = PROJECT_ROOT / "data:"
if not DATA_DIR.exists():
    DATA_DIR = PROJECT_ROOT / "data"

FIG_DIR = PROJECT_ROOT / "figures:"
if not FIG_DIR.exists():
    FIG_DIR = PROJECT_ROOT / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

RESULTS_DAY3_DIR = DATA_DIR / "results_day3"
RESULTS_DAY4_DIR = DATA_DIR / "results_day4"
RESULTS_DAY5_DIR = DATA_DIR / "results_day5"
RESULTS_DAY5_DIR.mkdir(parents=True, exist_ok=True)

print("Using data dir:", DATA_DIR)
print("Using figures dir:", FIG_DIR)
print("Using Day 5 results dir:", RESULTS_DAY5_DIR)

Using data dir: /Users/syedm/Desktop/MSAI/CSML/CSML FInal Project/data:
Using figures dir: /Users/syedm/Desktop/MSAI/CSML/CSML FInal Project/figures:
Using Day 5 results dir: /Users/syedm/Desktop/MSAI/CSML/CSML FInal Project/data:/results_day5


In [2]:
def eval_forecasts(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    return mae, rmse


def add_quarter(df):
    out = df.copy()
    out["Date"] = pd.to_datetime(out["Date"])  # safe if already datetime
    out["quarter"] = out["Date"].dt.to_period("Q").astype(str)
    return out


def metrics_by_quarter(df, pred_col, target_col="target_rv_10"):
    rows = []
    for q, g in df.groupby("quarter"):
        mae, rmse = eval_forecasts(g[target_col], g[pred_col])
        rows.append({"quarter": q, "MAE": mae, "RMSE": rmse, "n_obs": len(g)})
    return pd.DataFrame(rows).sort_values("quarter").reset_index(drop=True)


# Load test predictions from Day 3 / Day 4 outputs.
a_base = pd.read_csv(RESULTS_DAY3_DIR / "AAPL_test_with_predictions.csv", parse_dates=["Date"])
m_base = pd.read_csv(RESULTS_DAY3_DIR / "MSFT_test_with_predictions.csv", parse_dates=["Date"])
a_rf = pd.read_csv(RESULTS_DAY4_DIR / "AAPL_random_forest_test_predictions.csv", parse_dates=["Date"])
m_rf = pd.read_csv(RESULTS_DAY4_DIR / "MSFT_random_forest_test_predictions.csv", parse_dates=["Date"])
a_gb = pd.read_csv(RESULTS_DAY4_DIR / "AAPL_gradient_boosting_test_predictions.csv", parse_dates=["Date"])
m_gb = pd.read_csv(RESULTS_DAY4_DIR / "MSFT_gradient_boosting_test_predictions.csv", parse_dates=["Date"])

# Ensure quarter labels exist.
a_base = add_quarter(a_base)
m_base = add_quarter(m_base)
a_rf = add_quarter(a_rf)
m_rf = add_quarter(m_rf)
a_gb = add_quarter(a_gb)
m_gb = add_quarter(m_gb)

In [3]:
def quarterly_for_ticker(ticker, base_df, rf_df, gb_df):
    specs = [
        ("Naive", base_df, "pred_naive"),
        ("RollingMean10", base_df, "pred_rollmean"),
        ("LinearRegression", base_df, "pred_linear"),
        ("RandomForest", rf_df, "pred"),
        ("GradientBoosting", gb_df, "pred"),
    ]

    rows = []
    for model_name, df_model, pred_col in specs:
        qdf = metrics_by_quarter(df_model, pred_col)
        qdf["Ticker"] = ticker
        qdf["Model"] = model_name
        rows.append(qdf)

    out = pd.concat(rows, ignore_index=True)
    return out[["Ticker", "Model", "quarter", "n_obs", "MAE", "RMSE"]]


quarterly_metrics = pd.concat(
    [
        quarterly_for_ticker("AAPL", a_base, a_rf, a_gb),
        quarterly_for_ticker("MSFT", m_base, m_rf, m_gb),
    ],
    ignore_index=True,
)

model_order = ["Naive", "RollingMean10", "LinearRegression", "RandomForest", "GradientBoosting"]
quarterly_metrics["Model"] = pd.Categorical(
    quarterly_metrics["Model"], categories=model_order, ordered=True
)
quarterly_metrics = quarterly_metrics.sort_values(["Ticker", "quarter", "Model"]).reset_index(drop=True)

quarterly_metrics_rounded = quarterly_metrics.copy()
quarterly_metrics_rounded["MAE"] = quarterly_metrics_rounded["MAE"].round(5)
quarterly_metrics_rounded["RMSE"] = quarterly_metrics_rounded["RMSE"].round(5)

quarterly_metrics_rounded.head(15)

,Ticker,Model,quarter,n_obs,MAE,RMSE
0,AAPL,Naive,2023Q1,62,0.00118,0.00170
1,AAPL,RollingMean10,2023Q1,62,0.00325,0.00388
2,AAPL,LinearRegression,2023Q1,62,0.00132,0.00175
3,AAPL,RandomForest,2023Q1,62,0.00132,0.00178
4,AAPL,GradientBoosting,2023Q1,62,0.00132,0.00178
5,AAPL,Naive,2023Q2,62,0.00070,0.00156
6,AAPL,RollingMean10,2023Q2,62,0.00181,0.00288
7,AAPL,LinearRegression,2023Q2,62,0.00118,0.00176
8,AAPL,RandomForest,2023Q2,62,0.00081,0.00148
9,AAPL,GradientBoosting,2023Q2,62,0.00099,0.00158


In [4]:
# Final model performance table (test set) from Day 4 comparison.
perf = pd.read_csv(RESULTS_DAY4_DIR / "metrics_day4_comparison_test.csv")

paper_model_order = [
    "Naive",
    "RollingMean10",
    "LinearRegression",
    "DecisionTree",
    "RandomForest",
    "GradientBoosting",
]
perf["Model"] = pd.Categorical(perf["Model"], categories=paper_model_order, ordered=True)
perf = perf.sort_values(["Ticker", "Model"]).reset_index(drop=True)

model_performance_summary = perf.assign(
    MAE=lambda d: d["Test_MAE"].round(5),
    RMSE=lambda d: d["Test_RMSE"].round(5),
)[["Ticker", "Model", "MAE", "RMSE"]]

model_performance_summary.to_csv(RESULTS_DAY5_DIR / "model_performance_summary.csv", index=False)
quarterly_metrics_rounded.to_csv(RESULTS_DAY5_DIR / "quarterly_robustness_metrics.csv", index=False)

print("Saved:")
print(" -", (RESULTS_DAY5_DIR / "model_performance_summary.csv").name)
print(" -", (RESULTS_DAY5_DIR / "quarterly_robustness_metrics.csv").name)
model_performance_summary

Saved:
 - model_performance_summary.csv
 - quarterly_robustness_metrics.csv


,Ticker,Model,MAE,RMSE
0,AAPL,Naive,0.00096,0.00167
1,AAPL,RollingMean10,0.00248,0.00331
2,AAPL,LinearRegression,0.00123,0.00179
3,AAPL,DecisionTree,0.00133,0.00190
4,AAPL,RandomForest,0.00108,0.00168
5,AAPL,GradientBoosting,0.00116,0.00172
6,MSFT,Naive,0.00119,0.00207
7,MSFT,RollingMean10,0.00338,0.00433
8,MSFT,LinearRegression,0.00131,0.00202
9,MSFT,DecisionTree,0.00141,0.00220


In [5]:
# Recreate clean, consistently named final figures for paper use.
def build_day2_ready_df(path):
    df = pd.read_csv(path, parse_dates=["Date"])
    df["Date"] = pd.to_datetime(df["Date"], utc=True).dt.tz_localize(None)
    df = df.sort_values("Date").copy()
    df["log_return"] = np.log(df["Close"]).diff()
    df["rv_10"] = df["log_return"].rolling(10).std()
    return df


a_all = build_day2_ready_df(DATA_DIR / "AAPL_clean.csv")
m_all = build_day2_ready_df(DATA_DIR / "MSFT_clean.csv")

for ticker, df_all in [("AAPL", a_all), ("MSFT", m_all)]:
    # Price
    plt.figure(figsize=(9, 4))
    plt.plot(df_all["Date"], df_all["Close"])
    plt.title(f"{ticker} Close Price")
    plt.xlabel("Date")
    plt.ylabel("Price (USD)")
    plt.tight_layout()
    plt.savefig(FIG_DIR / f"{ticker}_price_timeseries.png", dpi=150)
    plt.close()

    # RV_10
    plt.figure(figsize=(9, 4))
    plt.plot(df_all["Date"], df_all["rv_10"])
    plt.title(f"{ticker} 10-day Realized Volatility")
    plt.xlabel("Date")
    plt.ylabel("10-day realized volatility")
    plt.tight_layout()
    plt.savefig(FIG_DIR / f"{ticker}_rv10_timeseries.png", dpi=150)
    plt.close()

# RF vs Linear vs Realized (test period)
def plot_model_comparison(tree_test_df, linear_df, ticker, model_label):
    plt.figure(figsize=(10, 4))
    plt.plot(tree_test_df["Date"], tree_test_df["target_rv_10"], label="Realized RV", linewidth=1.8)
    plt.plot(tree_test_df["Date"], tree_test_df["pred"], label=model_label, alpha=0.85)
    plt.plot(linear_df["Date"], linear_df["pred_linear"], label="Linear", alpha=0.75)
    plt.xlabel("Date")
    plt.ylabel("10-day realized volatility")
    plt.title(f"{ticker} - {model_label} vs Linear vs Realized (Test)")
    plt.legend()
    plt.tight_layout()
    plt.savefig(FIG_DIR / f"{ticker}_rf_vs_linear_test.png", dpi=150)
    plt.close()

plot_model_comparison(a_rf, a_base, "AAPL", "Random Forest")
plot_model_comparison(m_rf, m_base, "MSFT", "Random Forest")

print("Curated figures saved.")

Curated figures saved.


In [6]:
# Refit RF using best Day 4 params to inspect feature importances in this notebook.
import ast

feature_cols = [
    "rv_10",
    "log_close",
    "log_return_lag_1",
    "log_return_lag_2",
    "log_return_lag_3",
    "log_return_lag_4",
    "log_return_lag_5",
    "rv_10_lag_1",
    "log_volume",
    "volume_change",
]


def build_features_for_rf(path):
    df = pd.read_csv(path, parse_dates=["Date"])
    df["Date"] = pd.to_datetime(df["Date"], utc=True).dt.tz_localize(None)
    df = df.sort_values("Date").copy()
    df["log_return"] = np.log(df["Close"]).diff()
    df["rv_10"] = df["log_return"].rolling(10).std()
    df["target_rv_10"] = df["rv_10"].shift(-1)
    df["log_close"] = np.log(df["Close"])
    for k in range(1, 6):
        df[f"log_return_lag_{k}"] = df["log_return"].shift(k)
    df["rv_10_lag_1"] = df["rv_10"].shift(1)
    df["log_volume"] = np.log(df["Volume"].replace(0, np.nan))
    df["volume_change"] = df["Volume"].pct_change()
    return df.dropna().reset_index(drop=True)


def best_rf_params_for(ticker):
    tuning = pd.read_csv(RESULTS_DAY4_DIR / "tree_model_tuning_summary.csv")
    row = tuning[(tuning["Ticker"] == ticker) & (tuning["Model"] == "RandomForest")]
    if row.empty:
        return {"n_estimators": 200, "max_depth": None, "min_samples_leaf": 10, "n_jobs": -1, "random_state": 0}
    return ast.literal_eval(row.sort_values("Val_RMSE").iloc[0]["Params"])


def fit_rf_and_importance(ticker):
    df = build_features_for_rf(DATA_DIR / f"{ticker}_clean.csv")
    train = df[df["Date"] <= pd.Timestamp("2021-12-31")]
    params = best_rf_params_for(ticker)
    rf = RandomForestRegressor(**params)
    rf.fit(train[feature_cols], train["target_rv_10"])
    imp = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)
    return imp


a_imp = fit_rf_and_importance("AAPL")
m_imp = fit_rf_and_importance("MSFT")

print("AAPL top 10 features")
a_imp.head(10)

AAPL top 10 features


rv_10               0.941929
rv_10_lag_1         0.048940
log_volume          0.004293
log_return_lag_2    0.001187
log_return_lag_5    0.000959
log_return_lag_1    0.000697
volume_change       0.000620
log_close           0.000527
log_return_lag_3    0.000483
log_return_lag_4    0.000365
dtype: float64

In [7]:
print("MSFT top 10 features")
m_imp.head(10)

MSFT top 10 features


rv_10               0.943616
rv_10_lag_1         0.047871
log_volume          0.003702
log_close           0.001145
log_return_lag_1    0.000735
log_return_lag_3    0.000655
volume_change       0.000638
log_return_lag_5    0.000555
log_return_lag_2    0.000548
log_return_lag_4    0.000533
dtype: float64

In [8]:
# Draft key findings bullets for Results/Discussion notes.
def best_model_line(ticker):
    sub = model_performance_summary[model_performance_summary["Ticker"] == ticker].sort_values("RMSE")
    best = sub.iloc[0]
    return f"{ticker}: best test RMSE model is {best['Model']} (MAE={best['MAE']:.5f}, RMSE={best['RMSE']:.5f})."


def quarterly_stability_line(ticker):
    sub = quarterly_metrics_rounded[quarterly_metrics_rounded["Ticker"] == ticker]
    rf = sub[sub["Model"] == "RandomForest"]
    worst_q = rf.sort_values("RMSE", ascending=False).iloc[0]
    best_q = rf.sort_values("RMSE", ascending=True).iloc[0]
    return (
        f"{ticker}: RF is most accurate in {best_q['quarter']} (RMSE={best_q['RMSE']:.5f}) "
        f"and weakest in {worst_q['quarter']} (RMSE={worst_q['RMSE']:.5f})."
    )


def top_features_line(ticker, imp):
    top3 = ", ".join([f"{idx} ({val:.3f})" for idx, val in imp.head(3).items()])
    return f"{ticker}: top RF feature importances -> {top3}."


findings = [
    "Tree-based models and linear model are benchmarked against Naive and RollingMean10 baselines on 2023 test data.",
    best_model_line("AAPL"),
    best_model_line("MSFT"),
    quarterly_stability_line("AAPL"),
    quarterly_stability_line("MSFT"),
    top_features_line("AAPL", a_imp),
    top_features_line("MSFT", m_imp),
    "Volatility-based features (rv_10 and rv_10_lag_1) dominate feature importance, while volume-based features are typically weaker.",
]

for i, line in enumerate(findings, start=1):
    print(f"{i}. {line}")

(RESULTS_DAY5_DIR / "day5_key_findings.txt").write_text("\n".join(findings))
print("\nSaved notes:", RESULTS_DAY5_DIR / "day5_key_findings.txt")

1. Tree-based models and linear model are benchmarked against Naive and RollingMean10 baselines on 2023 test data.
2. AAPL: best test RMSE model is Naive (MAE=0.00096, RMSE=0.00167).
3. MSFT: best test RMSE model is RandomForest (MAE=0.00122, RMSE=0.00199).
4. AAPL: RF is most accurate in 2023Q4 (RMSE=0.00143) and weakest in 2023Q3 (RMSE=0.00189).
5. MSFT: RF is most accurate in 2023Q3 (RMSE=0.00160) and weakest in 2023Q2 (RMSE=0.00248).
6. AAPL: top RF feature importances -> rv_10 (0.942), rv_10_lag_1 (0.049), log_volume (0.004).
7. MSFT: top RF feature importances -> rv_10 (0.944), rv_10_lag_1 (0.048), log_volume (0.004).
8. Volatility-based features (rv_10 and rv_10_lag_1) dominate feature importance, while volume-based features are typically weaker.

Saved notes: /Users/syedm/Desktop/MSAI/CSML/CSML FInal Project/data:/results_day5/day5_key_findings.txt


### Day 5 Checklist (Implemented)

- Quarterly robustness metrics produced for Naive, RollingMean10, LinearRegression, RandomForest, and GradientBoosting.
- Final test-set summary table exported to `results_day5/model_performance_summary.csv`.
- Curated final figures saved with paper-friendly names (price, rv10, RF-vs-linear test plots).
- Feature-importance summaries generated for AAPL and MSFT.
- Key findings bullets generated and saved to `results_day5/day5_key_findings.txt`.